# 03c — Classical QUBO Solver & Comparison to Greedy Baseline

This notebook takes the QUBO built in `03b_qubo_design_for_trial_selection.ipynb`
and:

1. Loads the QUBO and the variable metadata.
2. Implements a simple classical solver for the QUBO using simulated annealing.
3. Compares the simulated annealing (SA) solution to:
   - The greedy baseline from `03a_scenario_selection_baseline.ipynb`.
4. Saves the SA solution as a reusable artifact for future quantum / hybrid
   experiments.

The goal is to have a concrete **classical benchmark** for the trial selection
problem that we can compare quantum or hybrid approaches against later.


In [1]:
# ============================================================
# Cell 1 — Load QUBO, metadata, and greedy baseline
# ============================================================

from pathlib import Path
import json
import pandas as pd

def log(msg: str) -> None:
    print(msg)

QUBO_PATH = Path("data/qubo/scenario_B_qubo.json")
META_PATH = Path("data/qubo/scenario_B_qubo_metadata.csv")
GREEDY_SEL_PATH = Path("data/scenarios/scenario_B_greedy_selection.csv")

# --- Load QUBO dictionary ---------------------------------------------------

if not QUBO_PATH.exists():
    raise FileNotFoundError(
        f"[Cell 1] Missing {QUBO_PATH}. Run 03b_qubo_design_for_trial_selection.ipynb first."
    )

with QUBO_PATH.open("r") as f:
    Q_serializable = json.load(f)

# Convert keys "i,j" -> (i, j) tuples
Q = {}
for key, val in Q_serializable.items():
    i_str, j_str = key.split(",")
    i, j = int(i_str), int(j_str)
    Q[(i, j)] = float(val)

log(f"[Cell 1] Loaded QUBO with {len(Q)} non-zero entries from {QUBO_PATH}")

# --- Load variable metadata -------------------------------------------------

if not META_PATH.exists():
    raise FileNotFoundError(
        f"[Cell 1] Missing {META_PATH}. Run 03b_qubo_design_for_trial_selection.ipynb first."
    )

meta_df = pd.read_csv(META_PATH)
log(f"[Cell 1] Loaded QUBO metadata with shape {meta_df.shape}")

# Ensure var_index is present
if "var_index" not in meta_df.columns:
    raise ValueError("[Cell 1] Metadata is missing 'var_index' column.")

N = meta_df["var_index"].nunique()
log(f"[Cell 1] Number of QUBO variables (N): {N}")

# --- Load greedy baseline selection (if present) ----------------------------

greedy_selection = None
if GREEDY_SEL_PATH.exists():
    greedy_selection = pd.read_csv(GREEDY_SEL_PATH)
    log(
        f"[Cell 1] Loaded greedy baseline selection "
        f"with shape {greedy_selection.shape}"
    )
else:
    log(f"[Cell 1] WARNING: {GREEDY_SEL_PATH} not found; continuing without greedy baseline.")

meta_df.head()


[Cell 1] Loaded QUBO with 820 non-zero entries from data/qubo/scenario_B_qubo.json
[Cell 1] Loaded QUBO metadata with shape (40, 14)
[Cell 1] Number of QUBO variables (N): 40
[Cell 1] Loaded greedy baseline selection with shape (1026, 14)


,var_index,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score
0,0,NCT00003042,Chemotherapy and Stem Cell Transplantation in ...,"Active, not recruiting",Phase 2,['Breast Cancer'],['filgrastim' 'cisplatin' 'cyclophosphamide' '...,['United States'],City of Hope Medical Center,City of Hope Medical Center,Global / Multi-Region,2.0,1.0,0.6
1,1,NCT06234631,Cannabidiol for Postoperative Opioid Reduction...,Recruiting,Phase 2,"['Knee Replacement Surgery' 'Osteoarthritis, K...",['Epidiolex oral solution' 'Placebo'],['United States'],Chad Brummett,Chad Brummett,Global / Multi-Region,2.0,1.0,0.6
2,2,NCT06257537,Sustained Acoustic Medicine for Symptomatic Tr...,Recruiting,Phase 2,['Osteo Arthritis Knee' 'Arthritis'],['Sustained Acoustic Device with 2.5% Diclofen...,['United States'],"ZetrOZ, Inc.","ZetrOZ, Inc.",Global / Multi-Region,2.0,1.0,0.6
3,3,NCT06257875,A Study to Assess Adverse Events and Change in...,"Active, not recruiting",Phase 2,['Ulcerative Colitis'],['Lutikizumab' 'Lutikizumab' 'Adalimumab'],['Australia' 'Austria' 'Belgium' 'Bulgaria' 'C...,AbbVie,AbbVie,Global / Multi-Region,2.0,1.0,0.6
4,4,NCT06259123,Neoadjuvant PSMA-RLT in Oligometastatic PCa,Recruiting,Phase 2,['Prostate Cancer'],['[177Lu]Lu-PSMA I&T'],['Austria'],Medical University of Vienna,Medical University of Vienna,Global / Multi-Region,2.0,1.0,0.6


### What Cell 1 Just Did

This step gathered all the inputs needed for classical QUBO solving and
comparison:

- Loaded the QUBO dictionary from:
  - `data/qubo/scenario_B_qubo.json`
  - Converted string keys `"i,j"` back into integer tuples `(i, j)`.
- Loaded the variable metadata from:
  - `data/qubo/scenario_B_qubo_metadata.csv`
  - Confirmed that `var_index` identifies each binary variable.
- Determined the number of QUBO variables `N` from the metadata.
- Optionally loaded the greedy baseline selection from:
  - `data/scenarios/scenario_B_greedy_selection.csv` (if present).

At this point:

- `Q` is the QUBO dictionary over `N` binary variables.
- `meta_df` maps `var_index` back to real trial attributes (e.g., `nct_id`,
  `estimated_trial_cost`, `benefit_score`).
- `greedy_selection` (if available) will be used later for comparison.


In [3]:
# ============================================================
# Cell 2 — QUBO energy function and simulated annealing solver
# ============================================================

import numpy as np
from typing import Dict, Tuple, List

def qubo_energy(Q: Dict[Tuple[int, int], float], x: np.ndarray) -> float:
    """
    Compute the QUBO energy for binary vector x, where x is a 1D NumPy array
    of 0/1 with length N, and Q is a dict mapping (i, j) to coefficient.
    """
    energy = 0.0
    # We assume Q includes both diagonal (i, i) and upper-triangular (i, j) terms.
    for (i, j), q_ij in Q.items():
        energy += q_ij * x[i] * x[j]
    return float(energy)


def simulated_annealing_qubo(
    Q: Dict[Tuple[int, int], float],
    N: int,
    num_steps: int = 5000,
    temp_start: float = 5.0,
    temp_end: float = 0.1,
    seed: int | None = 42,
) -> Tuple[np.ndarray, float]:
    """
    Simple simulated annealing for QUBO:
      - Start from a random 0/1 vector.
      - At each step, pick a random bit to flip.
      - Accept worse moves with probability exp(-(ΔE)/T).

    Returns:
      best_x:  binary vector with lowest energy found.
      best_E:  corresponding QUBO energy.
    """
    rng = np.random.default_rng(seed)

    # Random initial solution
    x = rng.integers(low=0, high=2, size=N, dtype=int)
    E = qubo_energy(Q, x)

    best_x = x.copy()
    best_E = E

    # Linear temperature schedule
    temps = np.linspace(temp_start, temp_end, num_steps)

    for step, T in enumerate(temps, start=1):
        # Pick a random index to flip
        i = rng.integers(low=0, high=N)
        x_new = x.copy()
        x_new[i] = 1 - x_new[i]  # flip bit

        E_new = qubo_energy(Q, x_new)
        dE = E_new - E

        # Metropolis acceptance rule
        if dE <= 0 or rng.random() < np.exp(-dE / T):
            x = x_new
            E = E_new

            if E < best_E:
                best_E = E
                best_x = x.copy()

        if step % 1000 == 0:
            log(f"[Cell 2] Step {step}/{num_steps}, current E={E:.4f}, best E={best_E:.4f}")

    log(f"[Cell 2] Finished SA: best energy found = {best_E:.4f}")
    return best_x, best_E


### What Cell 2 Just Did

This step defined the core machinery for classical QUBO optimization:

- `qubo_energy(Q, x)`:
  - Computes the QUBO objective value (energy) for a given binary vector `x`.
  - Sums `Q[(i, j)] * x[i] * x[j]` over all entries in the QUBO dictionary.

- `simulated_annealing_qubo(...)`:
  - Implements a simple simulated annealing (SA) algorithm:
    1. Starts from a random 0/1 vector of length `N`.
    2. Repeatedly picks a random bit to flip.
    3. Accepts improvements always, and accepts worse moves with probability
       `exp(-ΔE / T)` where `T` gradually decreases from `temp_start` to
       `temp_end`.
  - Tracks the lowest-energy solution seen (`best_x`, `best_E`) and returns it
    at the end of the run.

This solver is intentionally lightweight and dependency-free, but sufficient to
give us a concrete classical benchmark for the QUBO.


In [4]:
# ============================================================
# Cell 3 — Run SA, compute metrics, and compare to greedy
# ============================================================

# --- Run simulated annealing on the QUBO -----------------------------------

best_x, best_E = simulated_annealing_qubo(
    Q=Q,
    N=N,
    num_steps=5000,      # you can increase this later if desired
    temp_start=5.0,
    temp_end=0.1,
    seed=42,
)

# Convert solution to a DataFrame joined with metadata
meta_with_sel = meta_df.copy()
meta_with_sel["selected_sa"] = 0
meta_with_sel.loc[meta_with_sel["var_index"].isin(
    [i for i, bit in enumerate(best_x) if bit == 1]
), "selected_sa"] = 1

# Compute total cost and benefit for SA solution
if "estimated_trial_cost" not in meta_with_sel.columns:
    raise ValueError("[Cell 3] Metadata missing 'estimated_trial_cost' column.")

if "benefit_score" not in meta_with_sel.columns:
    raise ValueError("[Cell 3] Metadata missing 'benefit_score' column.")

sa_selected = meta_with_sel[meta_with_sel["selected_sa"] == 1].copy()

sa_total_cost = sa_selected["estimated_trial_cost"].sum()
sa_total_benefit = sa_selected["benefit_score"].sum()

log(f"[Cell 3] SA selected {len(sa_selected)} trials.")
log(f"[Cell 3] SA total cost:    {sa_total_cost:,.2f}")
log(f"[Cell 3] SA total benefit: {sa_total_benefit:.4f}")
log(f"[Cell 3] SA best QUBO energy: {best_E:.4f}")

display(
    sa_selected[
        ["var_index", "nct_id", "phase", "overall_status",
         "estimated_trial_cost", "benefit_score"]
    ].head(20)
)

# --- Compare to greedy baseline (if available) -----------------------------

if greedy_selection is not None:
    greedy_total_cost = greedy_selection["estimated_trial_cost"].sum()
    greedy_total_benefit = greedy_selection["benefit_score"].sum()

    log(f"[Cell 3] Greedy selected {len(greedy_selection)} trials.")
    log(f"[Cell 3] Greedy total cost:    {greedy_total_cost:,.2f}")
    log(f"[Cell 3] Greedy total benefit: {greedy_total_benefit:.4f}")

    # Compute overlap by nct_id, restricted to those in the QUBO candidate set
    if "nct_id" in greedy_selection.columns and "nct_id" in meta_with_sel.columns:
        sa_ids = set(sa_selected["nct_id"].dropna().astype(str))
        greedy_ids = set(greedy_selection["nct_id"].dropna().astype(str))

        overlap_ids = sa_ids.intersection(greedy_ids)
        overlap_count = len(overlap_ids)

        log(f"[Cell 3] Overlap in selected nct_id (SA ∩ Greedy): {overlap_count}")
        if len(sa_ids) > 0 and len(greedy_ids) > 0:
            jaccard = overlap_count / len(sa_ids.union(greedy_ids))
            log(f"[Cell 3] Jaccard similarity (by nct_id): {jaccard:.4f}")
else:
    log("[Cell 3] No greedy baseline loaded; skipping comparison.")


[Cell 2] Step 1000/5000, current E=-642.4000, best E=-642.4000
[Cell 2] Step 2000/5000, current E=-642.4000, best E=-642.4000
[Cell 2] Step 3000/5000, current E=-642.4000, best E=-642.4000
[Cell 2] Step 4000/5000, current E=-642.4000, best E=-642.4000
[Cell 2] Step 5000/5000, current E=-642.4000, best E=-642.4000
[Cell 2] Finished SA: best energy found = -642.4000
[Cell 3] SA selected 4 trials.
[Cell 3] SA total cost:    8.00
[Cell 3] SA total benefit: 2.4000
[Cell 3] SA best QUBO energy: -642.4000


,var_index,nct_id,phase,overall_status,estimated_trial_cost,benefit_score
2,2,NCT06257537,Phase 2,Recruiting,2.0,0.6
10,10,NCT06260709,Phase 2,"Active, not recruiting",2.0,0.6
15,15,NCT06261320,Phase 2,"Active, not recruiting",2.0,0.6
16,16,NCT06261359,Phase 2,Recruiting,2.0,0.6


[Cell 3] Greedy selected 1026 trials.
[Cell 3] Greedy total cost:    2,052.00
[Cell 3] Greedy total benefit: 615.6000
[Cell 3] Overlap in selected nct_id (SA ∩ Greedy): 4
[Cell 3] Jaccard similarity (by nct_id): 0.0039


### What Cell 3 Just Did

This step actually **ran** the classical QUBO solver and compared its solution
to the greedy baseline:

- Ran `simulated_annealing_qubo` on the QUBO `Q` with `N` variables.
- Constructed a metadata-joined view `meta_with_sel` and marked trials chosen
  by simulated annealing (`selected_sa = 1`).
- Computed:
  - Total SA-selected cost (`sa_total_cost`)
  - Total SA-selected benefit (`sa_total_benefit`)
  - Best QUBO energy (`best_E`)
- Displayed a sample of the SA-selected trials with `nct_id`, `phase`,
  `overall_status`, `estimated_trial_cost`, and `benefit_score`.

If the greedy baseline file was present, it also:

- Computed total greedy cost and total greedy benefit.
- Measured overlap between SA and greedy solutions by `nct_id`.
- Reported the Jaccard similarity between the two selected sets.

This gives us a concrete classical benchmark (greedy) and a classical QUBO
solver (simulated annealing) that we can compare future quantum results
against.

In [5]:
# ============================================================
# Cell 4 — Persist SA solution for future use
# ============================================================

SOLUTIONS_DIR = Path("data/qubo")
SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)

sa_solution_path = SOLUTIONS_DIR / "scenario_B_qubo_sa_solution.csv"
bitstring_path = SOLUTIONS_DIR / "scenario_B_qubo_sa_bitstring.txt"

# Save full metadata + selection flag
sa_output = meta_with_sel.copy()
sa_output.to_csv(sa_solution_path, index=False)
log(f"[Cell 4] Wrote SA solution with metadata to {sa_solution_path} "
    f"with shape {sa_output.shape}")

# Save a compact bitstring representation "010101..."
bitstring = "".join(str(int(bit)) for bit in best_x)
with bitstring_path.open("w") as f:
    f.write(bitstring + "\n")

log(f"[Cell 4] Wrote SA bitstring (length={len(bitstring)}) to {bitstring_path}")

[Cell 4] Wrote SA solution with metadata to data/qubo/scenario_B_qubo_sa_solution.csv with shape (40, 15)
[Cell 4] Wrote SA bitstring (length=40) to data/qubo/scenario_B_qubo_sa_bitstring.txt


### What Cell 4 Just Did

This step saved the simulated annealing solution in two complementary forms:

1. **Rich metadata table**:
   - `data/qubo/scenario_B_qubo_sa_solution.csv`
   - Contains:
     - `var_index` for each QUBO variable
     - Trial identifiers such as `nct_id`
     - Scenario features (phase, status, cost, benefit)
     - A `selected_sa` flag indicating which trials SA chose.

2. **Compact bitstring**:
   - `data/qubo/scenario_B_qubo_sa_bitstring.txt`
   - A single line of 0/1 characters encoding the SA solution.

These artifacts allow later notebooks (including quantum or hybrid ones) to:

- Compare their solutions to the SA baseline at the bitstring level, and
- Interpret which real-world clinical trials each bit corresponds to.


## Notebook Summary — Classical QUBO Solver & Greedy Comparison

In this notebook we:

1. Loaded the QUBO and variable metadata produced by `03b`.
2. Implemented a simple simulated annealing (SA) solver for the QUBO.
3. Ran SA to obtain a low-energy binary solution and mapped it back to
   clinical trials using the metadata.
4. Computed total cost and total benefit for the SA solution.
5. Compared SA’s selection (when available) to the greedy baseline from 03a:
   - Total cost and benefit,
   - Overlap in selected trials by `nct_id`.
6. Persisted the SA solution as:
   - A rich CSV (`scenario_B_qubo_sa_solution.csv`),
   - A compact bitstring (`scenario_B_qubo_sa_bitstring.txt`).

This gives us two strong **classical baselines** (greedy and SA) for the
scenario-based trial selection QUBO, ready for comparison with future
quantum or hybrid runs.
